In [1]:
# mediapipe version: 0.10.14
# python version: 3.10.11
#!pip install mediapipe==0.10.14
import cv2
import mediapipe as mp
import numpy as np
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

In [ ]:
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()
    cv2.imshow('Mediapipe Feed', frame)

    if cv2.waitKey(25) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
cap = cv2.VideoCapture(0)
# for a more accurate model, use higher confidence values, but then detection is finnicky.
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()

        # recolor to RGB because mediapipe expects RGB, but opencv feed is BGR by default.
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        # to save memory
        image.flags.writeable = False
        # make detections
        results = pose.process(image)
        # change back to true so that cv2 can write to it
        image.flags.writeable = True
        # recolor back to BGR
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # change the image that we're actually going to render
        # first drawing spec: landmarks (points) (color is in GBR)
        # second drawing spec: connections (lines)
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(255,255,0), thickness=2, circle_radius=2),
                                  mp_drawing.DrawingSpec(color=(255,0,255), thickness=2, circle_radius=2)
                                  )

        # render the image
        cv2.imshow('Mediapipe Feed', image)

        if cv2.waitKey(25) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


In [ ]:
mp_drawing.DrawingSpec??

In [ ]:
# determining joints

cap = cv2.VideoCapture(0)
# for a more accurate model, use higher confidence values, but then detection is finnicky.
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()

        # recolor to RGB because mediapipe expects RGB, but opencv feed is BGR by default.
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        # to save memory
        image.flags.writeable = False
        # make detections
        results = pose.process(image)
        # change back to true so that cv2 can write to it
        image.flags.writeable = True
        # recolor back to BGR
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark
            print(landmarks)
        except:
            pass




        # change the image that we're actually going to render
        # first drawing spec: landmarks (points) (color is in GBR)
        # second drawing spec: connections (lines)
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(255,255,0), thickness=2, circle_radius=2),
                                  mp_drawing.DrawingSpec(color=(255,0,255), thickness=2, circle_radius=2)
                                  )

        # render the image
        cv2.imshow('Mediapipe Feed', image)

        if cv2.waitKey(25) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


In [ ]:
len(landmarks)

In [ ]:
# mp_pose.PoseLandmark is a list of labels.
# mp_pose.PoseLandmark.LEFT_SHOULDER.value gives us the value or index of the left shoulder
for lndmark in mp_pose.PoseLandmark:
    print(lndmark)

In [ ]:
print(landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value])
print(landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value])
print(landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value])

In [2]:
# Calculate Angles

def calculate_angle(a,b,c):
    a = np.array(a) # first
    b = np.array(b) # mid
    c = np.array(c) # end

    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0 / np.pi)
    if angle > 180.0:
        angle = 360-angle
    
    return angle


In [3]:
shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]

calculate_angle(shoulder, elbow, wrist)

NameError: name 'landmarks' is not defined

In [4]:
# determining joints

cap = cv2.VideoCapture(0)
# for a more accurate model, use higher confidence values, but then detection is finnicky.
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()

        # recolor to RGB because mediapipe expects RGB, but opencv feed is BGR by default.
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        # to save memory
        image.flags.writeable = False
        # make detections
        results = pose.process(image)
        # change back to true so that cv2 can write to it
        image.flags.writeable = True
        # recolor back to BGR
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark

            # get coordinates
            shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
            elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
            wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]
            
            # calc angle
            angle = calculate_angle(shoulder, elbow, wrist)
            # visualize angle
            # MULTIPLY BY DIMENSIONS OF WEBCAM FEED
            cv2.putText(image, str(angle),
                        tuple(np.multiply(elbow, [640, 480]).astype(int)),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA
                        )
            
        except:
            pass




        # change the image that we're actually going to render
        # first drawing spec: landmarks (points) (color is in GBR)
        # second drawing spec: connections (lines)
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(255,255,0), thickness=2, circle_radius=2),
                                  mp_drawing.DrawingSpec(color=(255,0,255), thickness=2, circle_radius=2)
                                  )

        # render the image
        cv2.imshow('Mediapipe Feed', image)

        if cv2.waitKey(25) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


C:\Users\nosha\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [5]:
# Calculate Angles

def calculate_angle(a,b,c):
    a = np.array(a) # first
    b = np.array(b) # mid
    c = np.array(c) # end

    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians*180.0 / np.pi)
    if angle > 180.0:
        angle = 360-angle
    
    return angle

## CURL COUNTER ##

counter = 0
stage = None

cap = cv2.VideoCapture(0)
# for a more accurate model, use higher confidence values, but then detection is finnicky.
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()

        # recolor to RGB because mediapipe expects RGB, but opencv feed is BGR by default.
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        # to save memory
        image.flags.writeable = False
        # make detections
        results = pose.process(image)
        # change back to true so that cv2 can write to it
        image.flags.writeable = True
        # recolor back to BGR
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # extract landmarks
        try:
            landmarks = results.pose_landmarks.landmark

            # get coordinates
            shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
            elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
            wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x, landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]
            
            # calc angle
            angle = calculate_angle(shoulder, elbow, wrist)
            # visualize angle
            # MULTIPLY BY DIMENSIONS OF WEBCAM FEED
            cv2.putText(image, str(angle),
                        tuple(np.multiply(elbow, [640, 480]).astype(int)),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA
                        )
            
            # curl counter logic:
            if angle > 160:
                stage = "down"
            if angle < 30 and stage =="down":
                stage = "up"
                counter += 1
                print(counter)

        except:
            pass

        # render curl counter
        # setup status box
        cv2.rectangle(image, (0,0), (255,72), (255,0,255), -1)
        cv2.putText(image, "REPS", (15,12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,0), 1, cv2.LINE_AA)
        cv2.putText(image, str(counter), (10,60), cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,0), 2, cv2.LINE_AA)

        cv2.putText(image, "STAGE", (65,12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,0), 1, cv2.LINE_AA)
        cv2.putText(image, stage, (60,60), cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,0), 2, cv2.LINE_AA)

        # change the image that we're actually going to render
        # first drawing spec: landmarks (points) (color is in BGR)
        # second drawing spec: connections (lines)
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color=(255,255,0), thickness=2, circle_radius=2),
                                  mp_drawing.DrawingSpec(color=(255,0,255), thickness=2, circle_radius=2)
                                  )

        # render the image
        cv2.imshow('Mediapipe Feed', image)

        if cv2.waitKey(25) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41


In [ ]:
cv2.rectangle??